In [ ]:
# load packages
import numpy as np
import pandas as pd
import impectPy
from tqdm import tqdm

In [ ]:
# set login credentials
username = "yourUsername"
password = "yourPassword"

# create Impect instance and login
api = impectPy.Impect()
api.login(username=username, password=password)

In [ ]:
# set simulation parameters
iteration = 1385  # iterationId of the competition to simulate
simulation_date = "2026-04-27"  # date from which to start simulating
simulation_runs = 10000  # number of Monte Carlo simulation runs
expected_matches = 306  # total matches in the iteration (raises error if mismatch)
use_negbin = True  # True for Negative Binomial, False for Poisson distribution

In [ ]:
# fetch matches for iteration
matches = api.getMatches(iteration=iteration)

# check match count against expected
if len(matches) != expected_matches:
    raise ValueError(
        f"Expected {expected_matches} matches but found {len(matches)}. "
        f"Check the iterationId or update expected_matches."
    )

# show matches dataframe
matches.head()

In [ ]:
# fetch prediction model coefficients
coefficients = api.getSquadCoefficients(iteration=iteration)

# show coefficients
coefficients.head()

In [ ]:
# fetch match scores via squad matchsums
match_ids = matches["id"].tolist()
matchsums = api.getSquadMatchsums(matches=match_ids)

# show matchsums dataframe
matchsums.head()

In [ ]:
# prepare match scores by adjusting for own goals
# a team's final score = their GOALS + opponent's OWNGOALS
matchsums_with_sides = matchsums[["matchId", "squadId", "GOALS", "OWNGOALS"]].copy()

# identify home/away squads per match
match_sides = matches[["id", "homeSquadId", "awaySquadId"]].rename(
    columns={"id": "matchId"}
)
matchsums_with_sides = matchsums_with_sides.merge(match_sides, on="matchId")

# get opponent's own goals
matchsums_with_sides["opponentOwnGoals"] = np.where(
    matchsums_with_sides["squadId"] == matchsums_with_sides["homeSquadId"],
    matchsums_with_sides.merge(
        matchsums_with_sides[["matchId", "squadId", "OWNGOALS"]],
        left_on=["matchId", "awaySquadId"],
        right_on=["matchId", "squadId"],
        suffixes=("", "_opp"),
    )["OWNGOALS_opp"],
    matchsums_with_sides.merge(
        matchsums_with_sides[["matchId", "squadId", "OWNGOALS"]],
        left_on=["matchId", "homeSquadId"],
        right_on=["matchId", "squadId"],
        suffixes=("", "_opp"),
    )["OWNGOALS_opp"],
)

# adjusted goals = team's goals + opponent's own goals
matchsums_with_sides["adjustedGoals"] = (
    matchsums_with_sides["GOALS"] + matchsums_with_sides["opponentOwnGoals"].fillna(0)
)

# pivot to home/away scores
home_scores = matchsums_with_sides[
    matchsums_with_sides["squadId"] == matchsums_with_sides["homeSquadId"]
][["matchId", "adjustedGoals"]].rename(columns={"adjustedGoals": "homeGoals"})

away_scores = matchsums_with_sides[
    matchsums_with_sides["squadId"] == matchsums_with_sides["awaySquadId"]
][["matchId", "adjustedGoals"]].rename(columns={"adjustedGoals": "awayGoals"})

scores = home_scores.merge(away_scores, on="matchId", how="outer")

# merge scores with matches
matches = matches.merge(scores, left_on="id", right_on="matchId", how="left")
print(f"Matches with scores: {matches['homeGoals'].notna().sum()}/{len(matches)}.")

In [ ]:
# prepare dates
matches["date"] = (
    pd.to_datetime(matches["scheduledDate"]).dt.tz_localize(None).dt.normalize()
)
coefficients["date"] = pd.to_datetime(coefficients["date"]).dt.normalize()
sim_date = pd.to_datetime(simulation_date)

# validate simulation date is not in the future
today = pd.to_datetime("today").normalize()
if sim_date > today:
    raise ValueError(
        f"Simulation date {sim_date.date()} cannot be in the future."
    )

# validate simulation date is within the match date range
min_date = matches["date"].min()
max_date = matches["date"].max()
if not (min_date <= sim_date <= max_date):
    raise ValueError(
        f"Simulation date {sim_date.date()} is outside available range "
        f"{min_date.date()} to {max_date.date()}."
    )

In [ ]:
# select most recent coefficient date <= simulation date
coefficients["squadId"] = coefficients["squadId"].astype("int64")
coeff_dates = sorted(coefficients["date"].unique())
valid_dates = [d for d in coeff_dates if d <= sim_date]
if not valid_dates:
    raise ValueError(f"No coefficients available for date {sim_date.date()}.")
coeff_date = valid_dates[-1]
print(f"Using model coefficients from {coeff_date.date()}.")

# filter to the selected coefficient date
coeff_snapshot = coefficients[coefficients["date"] == coeff_date].copy()

# extract competition-level coefficients
intercept = coeff_snapshot["interceptCoefficient"].iloc[0]
home_coeff = coeff_snapshot["homeCoefficient"].iloc[0]
comp_coeff = coeff_snapshot["competitionCoefficient"].iloc[0]

# extract squad-level coefficients as lookup dicts
attack_coeffs = coeff_snapshot.set_index("squadId")["attackCoefficient"].to_dict()
defense_coeffs = coeff_snapshot.set_index("squadId")["defenseCoefficient"].to_dict()

In [ ]:
# compute expected goals using the Poisson regression formula
matches["homeSquadId"] = matches["homeSquadId"].astype("int64")
matches["awaySquadId"] = matches["awaySquadId"].astype("int64")

matches["attackHome"] = matches["homeSquadId"].map(attack_coeffs)
matches["defenseHome"] = matches["homeSquadId"].map(defense_coeffs)
matches["attackAway"] = matches["awaySquadId"].map(attack_coeffs)
matches["defenseAway"] = matches["awaySquadId"].map(defense_coeffs)

matches["predHome"] = np.exp(
    intercept + home_coeff + comp_coeff
    + matches["attackHome"] + matches["defenseAway"]
)
matches["predAway"] = np.exp(
    intercept + comp_coeff
    + matches["attackAway"] + matches["defenseHome"]
)

# check for matches missing predictions
missing_preds = matches["predHome"].isna().sum()
if missing_preds > 0:
    print(f"Warning: {missing_preds} matches missing predictions.")

# show matches with predictions
matches[["homeSquadName", "awaySquadName", "date", "homeGoals", "awayGoals", "predHome", "predAway"]].head()

In [ ]:
# identify matches needing simulation (future or postponed without scores)
needs_simulation = (
    (matches["date"] >= sim_date)
    | matches["homeGoals"].isna()
    | matches["awayGoals"].isna()
)

# warn about past matches missing scores that will be simulated
past_missing_mask = (
    (matches["date"] < sim_date)
    & (matches["homeGoals"].isna() | matches["awayGoals"].isna())
)
if past_missing_mask.any():
    print(
        f"Warning: {past_missing_mask.sum()} past matches missing scores "
        f"will be simulated."
    )

future_matches = matches[needs_simulation].copy()
played_matches = matches[~needs_simulation].copy()

if future_matches.empty:
    raise ValueError("No matches found to simulate.")

# log simulation info
distribution_info = (
    "Negative Binomial (overdispersion=0.15)" if use_negbin else "Poisson"
)
print(
    f"Played matches: {len(played_matches)}, "
    f"matches to simulate: {len(future_matches)}."
)
print(f"Distribution: {distribution_info}.")

In [ ]:
# run Monte Carlo simulations
all_teams = set(matches["homeSquadName"].unique()) | set(
    matches["awaySquadName"].unique()
)
num_teams = len(all_teams)
simulation_results = []

for run in tqdm(range(simulation_runs), desc="Simulating"):
    np.random.seed(run)

    # simulate future match goals
    pred_home = future_matches["predHome"].values
    pred_away = future_matches["predAway"].values

    if use_negbin:
        n_param = 1.0 / 0.15
        sim_home_goals = np.random.negative_binomial(
            n_param, n_param / (n_param + pred_home)
        )
        sim_away_goals = np.random.negative_binomial(
            n_param, n_param / (n_param + pred_away)
        )
    else:
        sim_home_goals = np.random.poisson(pred_home)
        sim_away_goals = np.random.poisson(pred_away)

    # create simulated matches DataFrame
    sim_future = future_matches.copy()
    sim_future["homeGoals"] = sim_home_goals
    sim_future["awayGoals"] = sim_away_goals

    # combine played and simulated matches
    sim_all = pd.concat(
        [played_matches, sim_future[played_matches.columns]], ignore_index=True
    )

    # calculate standings for this run
    team_stats = {}
    actual_stats = {}
    squad_id_mapping = {}

    for _, match in sim_all.iterrows():
        home = match["homeSquadName"]
        away = match["awaySquadName"]
        home_goals = match["homeGoals"]
        away_goals = match["awayGoals"]

        squad_id_mapping[home] = int(match["homeSquadId"])
        squad_id_mapping[away] = int(match["awaySquadId"])

        if pd.isna(home_goals) or pd.isna(away_goals):
            continue

        home_goals = int(home_goals)
        away_goals = int(away_goals)

        # initialize team stats
        for team in [home, away]:
            if team not in team_stats:
                team_stats[team] = {
                    "points": 0,
                    "goals_for": 0,
                    "goals_against": 0,
                    "matches": 0,
                }
            if team not in actual_stats:
                actual_stats[team] = {
                    "points": 0,
                    "goals_for": 0,
                    "goals_against": 0,
                    "matches": 0,
                }

        # update combined standings (actual + simulated)
        team_stats[home]["goals_for"] += home_goals
        team_stats[home]["goals_against"] += away_goals
        team_stats[home]["matches"] += 1
        team_stats[away]["goals_for"] += away_goals
        team_stats[away]["goals_against"] += home_goals
        team_stats[away]["matches"] += 1

        if home_goals > away_goals:
            team_stats[home]["points"] += 3
        elif home_goals < away_goals:
            team_stats[away]["points"] += 3
        else:
            team_stats[home]["points"] += 1
            team_stats[away]["points"] += 1

        # update actual standings (played matches only)
        if match["date"] < sim_date:
            actual_stats[home]["goals_for"] += home_goals
            actual_stats[home]["goals_against"] += away_goals
            actual_stats[home]["matches"] += 1
            actual_stats[away]["goals_for"] += away_goals
            actual_stats[away]["goals_against"] += home_goals
            actual_stats[away]["matches"] += 1

            if home_goals > away_goals:
                actual_stats[home]["points"] += 3
            elif home_goals < away_goals:
                actual_stats[away]["points"] += 3
            else:
                actual_stats[home]["points"] += 1
                actual_stats[away]["points"] += 1

    # build run results
    run_df = pd.DataFrame.from_dict(team_stats, orient="index")
    run_df["goal_diff"] = run_df["goals_for"] - run_df["goals_against"]
    run_df = run_df.rename(
        columns={
            "points": "points_proj",
            "goals_for": "gf_proj",
            "goals_against": "ga_proj",
            "goal_diff": "gd_proj",
            "matches": "matches_proj",
        }
    )

    # add actual stats
    actual_df = pd.DataFrame.from_dict(actual_stats, orient="index")
    actual_df["goal_diff"] = actual_df["goals_for"] - actual_df["goals_against"]
    run_df["points"] = actual_df["points"]
    run_df["goals_for"] = actual_df["goals_for"]
    run_df["goals_against"] = actual_df["goals_against"]
    run_df["goal_diff"] = actual_df["goal_diff"]
    run_df["matches"] = actual_df["matches"]

    # rank by projected stats
    run_df = run_df.sort_values(
        ["points_proj", "gd_proj", "gf_proj"], ascending=[False, False, False]
    )
    run_df["rank"] = range(1, len(run_df) + 1)
    run_df["squadId"] = run_df.index.map(squad_id_mapping)
    run_df.index.name = "squad"
    simulation_results.append(run_df.reset_index())

In [ ]:
# aggregate results across all simulations
all_results = pd.concat(simulation_results, ignore_index=True)

# average projected stats across simulations
avg_stats = (
    all_results.groupby("squad")
    .agg(
        squadId=("squadId", "first"),
        matches_played=("matches", "first"),
        points_actual=("points", "first"),
        gf_actual=("goals_for", "first"),
        ga_actual=("goals_against", "first"),
        gd_actual=("goal_diff", "first"),
        points_proj=("points_proj", "mean"),
        gf_proj=("gf_proj", "mean"),
        ga_proj=("ga_proj", "mean"),
        gd_proj=("gd_proj", "mean"),
    )
    .round(1)
)

# calculate rank probabilities
for rank in range(1, num_teams + 1):
    all_results[f"rank_{rank}"] = (all_results["rank"] == rank).astype(int)

rank_cols = [f"rank_{i}" for i in range(1, num_teams + 1)]
rank_probs = all_results.groupby("squad")[rank_cols].mean().round(3)
rank_probs.columns = pd.Index([f"rank_{i}_prob" for i in range(1, num_teams + 1)])

# combine results
final = avg_stats.join(rank_probs)
final = final.sort_values(["points_proj", "gd_proj", "gf_proj"] , ascending=False).reset_index()

# show final standings
final